In [ ]:
import time
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from models.gpt2_jax import GPT2JAX


def prepare_dataset(tokenizer, split="train", seq_len=256):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")[split]

    def encode(ex):
        tok = tokenizer(
            ex["text"],
            truncation=True,
            padding="max_length",
            max_length=seq_len,
        )
        return {
            "input_ids": tok["input_ids"],
            "attention_mask": tok["attention_mask"],
        }

    ds = ds.map(encode, batched=True, remove_columns=["text"],
                load_from_cache_file=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
    return ds


def cross_entropy_loss(logits, labels):
    logits = logits[:, :-1, :]   # (B, T-1, vocab)
    labels = labels[:, 1:]       # (B, T-1)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    return loss.mean()


@jax.jit
def train_step(state, input_ids):
    def loss_fn(params):
        logits = state.apply_fn(
            {"params": params},
            input_ids,
            training=True,
            rngs={"dropout": jax.random.PRNGKey(0)},
        )
        return cross_entropy_loss(logits, input_ids)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


def tpu_mem_str():
    mem = jax.devices()[0].memory_stats()
    if mem:
        used = mem["bytes_in_use"] / 1024**3
        total = mem["bytes_limit"] / 1024**3
        return f"mem={used:.2f}/{total:.2f}GB"
    return "mem=n/a"


def train_single_tpu(num_epochs=1, bs=16, lr=2e-5):
    print(f"devices: {jax.devices()}")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = GPT2JAX()
    key = jax.random.PRNGKey(0)
    dummy = jnp.ones((1, 256), dtype=jnp.int32)
    params = model.init(key, dummy)["params"]

    total_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
    print(f"GPT-2 parameters: {total_params/1e6:.1f}M")
    print(f"after init: {tpu_mem_str()}")

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(lr, eps=1e-5),
    )
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=params, tx=tx
    )

    loader = DataLoader(
        prepare_dataset(tokenizer, "train"),
        batch_size=bs,
        shuffle=True,
        num_workers=0,
    )

    print(f"steps per epoch: {len(loader)}")
    print("compiling train_step on first batch — takes 2-5 min, do not panic...")

    for epoch in range(num_epochs):
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            input_ids = jnp.array(batch["input_ids"].numpy())

            state, loss = train_step(state, input_ids)

            if i == 0:
                jax.block_until_ready(loss)
                compile_time = time.perf_counter() - epoch_start
                print(f"step 0 done | JIT compile time: {compile_time:.1f}s | "
                      f"loss={loss:.4f} | {tpu_mem_str()}")
                # reset timers after compilation so they don't skew throughput
                epoch_start = time.perf_counter()
                window_start = time.perf_counter()
                window_samples = 0
                continue

            window_samples += bs

            if i % 20 == 0:
                jax.block_until_ready(loss)
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss:.4f} "
                    f"throughput={throughput:.1f} samples/s | {tpu_mem_str()}"
                )

                window_start = time.perf_counter()
                window_samples = 0

        jax.block_until_ready(loss)
        epoch_time = time.perf_counter() - epoch_start
        # steps after step 0 = len(loader) - 1, each processing bs samples
        real_samples = (len(loader) - 1) * bs
        avg_throughput = real_samples / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s | "
            f"{tpu_mem_str()} ===\n"
        )


if __name__ == "__main__":
    train_single_tpu(num_epochs=1, bs=16, lr=2e-5)

In [ ]:
!git -C /content/training pull 2>/dev/null || git clone https://github.com/krishnajha23/training.git /content/training
%cd /content/training
!ls models/

In [ ]:
import jax
import jax.numpy as jnp
import flax
import optax

print(f"JAX version: {jax.__version__}")
print(f"devices: {jax.devices()}")
print(f"device count: {jax.device_count()}")

# confirm it can actually compute
x = jnp.ones((4, 256, 768))
print(f"test tensor device: {x.devices()}")
print(f"test tensor dtype: {x.dtype}")
print(f"flax: {flax.__version__}")
print(f"optax: {optax.__version__}")

In [ ]:
import jax
import jax.numpy as jnp

for device in jax.devices():
    mem_stats = device.memory_stats()
    if mem_stats:
        limit_gb = mem_stats["bytes_limit"] / 1024**3
        print(f"{device}: {limit_gb:.2f}GB total")

In [ ]:
from models.gpt2_jax import GPT2JAX

model = GPT2JAX()
key = jax.random.PRNGKey(0)
dummy = jnp.ones((1, 256), dtype=jnp.int32)
params = model.init(key, dummy)["params"]

total_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"params: {total_params/1e6:.1f}M")

# check memory after just loading params
mem_stats = jax.devices()[0].memory_stats()
print(f"after init: {mem_stats['bytes_in_use']/1024**3:.2f}GB")

# try a single forward pass at different batch sizes
for bs in [1, 4, 8, 16]:
    try:
        x = jnp.ones((bs, 256), dtype=jnp.int32)
        out = model.apply({"params": params}, x)
        mem = jax.devices()[0].memory_stats()["bytes_in_use"] / 1024**3
        print(f"bs={bs}: ok, mem={mem:.2f}GB")
    except Exception as e:
        print(f"bs={bs}: OOM")
        break

In [ ]:
import time
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state
from transformers import GPT2Tokenizer
from datasets import load_dataset
print("imports ok")

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

from torch.utils.data import DataLoader

ds = load_dataset("wikitext", "wikitext-2-raw-v1")["train"]
def encode(ex):
    tok = tokenizer(ex["text"], truncation=True, padding="max_length", max_length=256)
    return {"input_ids": tok["input_ids"]}
ds = ds.map(encode, batched=True, remove_columns=["text"], load_from_cache_file=True)
ds.set_format(type="torch", columns=["input_ids"])
loader = DataLoader(ds, batch_size=16, shuffle=True, num_workers=0)

batch = next(iter(loader))
print(f"batch shape: {batch['input_ids'].shape}")
print("dataloader ok")

In [ ]:
# cell 3 — small model to verify pipeline
from models.gpt2_jax import GPT2JAX
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state

# 4 layers, 4 heads, 256 embd — ~10M params, much cheaper to compile
model = GPT2JAX(n_layer=4, n_head=4, n_embd=256)
key = jax.random.PRNGKey(0)
params = model.init(key, jnp.ones((1, 256), dtype=jnp.int32))["params"]

total_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"params: {total_params/1e6:.1f}M")

tx = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(2e-5, eps=1e-5))
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

input_ids = jnp.array(batch["input_ids"].numpy())

@jax.jit
def train_step(state, input_ids):
    def loss_fn(params):
        logits = state.apply_fn(
            {"params": params}, input_ids,
            training=True, rngs={"dropout": jax.random.PRNGKey(0)}
        )
        logits = logits[:, :-1, :]
        labels = input_ids[:, 1:]
        return optax.softmax_cross_entropy_with_integer_labels(logits, labels).mean()
    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

print("compiling... (takes 1-2 min for small model)")
state, loss = train_step(state, input_ids)
jax.block_until_ready(loss)
print(f"loss: {loss:.4f} — pipeline works")

In [ ]:
import time
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state
from torch.utils.data import Dataset, DataLoader
import torch
from models.two_tower_jax import TwoTowerModelJAX
from models.loss_jax import in_batch_contrastive_loss_jax


class SyntheticTwoTowerDataset(Dataset):
    """
    Synthetic dataset for two-tower benchmarking.
    Same dims as Amazon Reviews preprocessed features:
      user_feature_dim = 32 (embedding) + 2 (stats) = 34
      item_feature_dim = 32 (embedding) + 2 (stats) = 34
    """
    def __init__(self, num_samples=50000, user_feature_dim=34, item_feature_dim=34):
        self.num_samples = num_samples
        self.user_features = torch.randn(num_samples, user_feature_dim)
        self.item_features = torch.randn(num_samples, item_feature_dim)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return {
            "user_features": self.user_features[idx],
            "item_features": self.item_features[idx],
        }


@jax.jit
def train_step(state, user_features, item_features):
    def loss_fn(params):
        user_emb, item_emb = state.apply_fn(
            {"params": params},
            user_features,
            item_features,
            training=True,
            rngs={"dropout": jax.random.PRNGKey(0)},
        )
        return in_batch_contrastive_loss_jax(user_emb, item_emb)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


def tpu_mem_str():
    mem = jax.devices()[0].memory_stats()
    if mem:
        used = mem["bytes_in_use"] / 1024**3
        total = mem["bytes_limit"] / 1024**3
        return f"mem={used:.2f}/{total:.2f}GB"
    return "mem=n/a"


def train_two_tower_tpu(
    num_epochs=1,
    bs=256,
    lr=1e-3,
    num_samples=50000,
    user_feature_dim=34,
    item_feature_dim=34,
    hidden_dims=(256, 128),
    embed_dim=64,
):
    print(f"devices: {jax.devices()}")

    model = TwoTowerModelJAX(
        user_feature_dim=user_feature_dim,
        item_feature_dim=item_feature_dim,
        hidden_dims=hidden_dims,
        embed_dim=embed_dim,
    )

    key = jax.random.PRNGKey(0)
    dummy_user = jnp.ones((1, user_feature_dim))
    dummy_item = jnp.ones((1, item_feature_dim))
    params = model.init(key, dummy_user, dummy_item)["params"]

    total_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
    print(f"two-tower parameters: {total_params/1e6:.2f}M")
    print(f"after init: {tpu_mem_str()}")

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(lr, eps=1e-5),
    )
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=params, tx=tx
    )

    dataset = SyntheticTwoTowerDataset(num_samples, user_feature_dim, item_feature_dim)
    loader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=True,
        num_workers=0,
    )

    print(f"steps per epoch: {len(loader)}")
    print("compiling train_step on first batch...")

    total_samples_seen = 0

    for epoch in range(num_epochs):
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            user_features = jnp.array(batch["user_features"].numpy())
            item_features = jnp.array(batch["item_features"].numpy())

            state, loss = train_step(state, user_features, item_features)

            if i == 0:
                jax.block_until_ready(loss)
                compile_time = time.perf_counter() - epoch_start
                print(f"step 0 done | JIT compile time: {compile_time:.1f}s | "
                      f"loss={loss:.4f} | {tpu_mem_str()}")
                epoch_start = time.perf_counter()
                window_start = time.perf_counter()
                window_samples = 0
                continue

            window_samples += bs
            total_samples_seen += bs

            if i % 20 == 0:
                jax.block_until_ready(loss)
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss:.4f} "
                    f"throughput={throughput:.1f} samples/s | {tpu_mem_str()}"
                )

                window_start = time.perf_counter()
                window_samples = 0

        jax.block_until_ready(loss)
        epoch_time = time.perf_counter() - epoch_start
        avg_throughput = total_samples_seen / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s | "
            f"{tpu_mem_str()} ===\n"
        )


if __name__ == "__main__":
    train_two_tower_tpu(num_epochs=1, bs=256, lr=1e-3)